In [1]:
import warnings
# 忽略所有 DeprecationWarning
warnings.filterwarnings("ignore", category=DeprecationWarning, module="scib|scgen|typing_extensions")

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import os
from scipy.sparse import issparse
import scvi
import scib
from scarches.models.scpoli import scPoli
import scarches as sca
from harmony import harmonize
import pyliger
from scarches.dataset.trvae.data_handling import remove_sparsity
import dask.dataframe as dd
import time
os.getcwd()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
 captum (see https://github.com/pytorch/captum).
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/_settings.py:450: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  IPython.display.set_matplotlib_formats(*ipython_format)
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/umap/__init__.py:9: ImportWarning: T

'/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration'

In this notebook the annotated datasets are concatinated, a ensembl <-> gene name mapping dictionary is created and different integration methods are implemented/executed on the dataset

# Mapping file for ensembl genes

# 1. Load and inspect the data

#### Abdominal Aortic Aneurysm (AAA)

In [84]:
path_1_JD = os.path.join("../../output_data/measure_data/1-JD/output/1-JD_annot_add.h5ad")
path_2_ZDZJ = os.path.join("../../output_data/measure_data/2-ZDZJ/output/2-ZDZJ_annot_add.h5ad")
path_AAA = os.path.join("../../output_data/measure_data/AAA/output/AAA_annot_add.h5ad")
path_AAA_1_3LIB = os.path.join( "../../output_data/measure_data/AAA-1-3LIB/AAA-1-3LIB_annot_add.h5ad")
path_AAA_8 = os.path.join("../../output_data/measure_data/AAA-8/output/AAA-8_annot_add.h5ad")
path_AAA_D = os.path.join("../../output_data/measure_data/AAA-D/output/AAA-D_annot_add.h5ad")

In [85]:
data_1_JD= sc.read_h5ad(path_1_JD)
data_2_ZDZJ = sc.read_h5ad(path_2_ZDZJ)
data_AAA = sc.read_h5ad(path_AAA)
data_AAA_1_3LIB = sc.read_h5ad(path_AAA_1_3LIB)
data_AAA_8 = sc.read_h5ad(path_AAA_8)
data_AAA_D = sc.read_h5ad(path_AAA_D)

In [86]:
data_1_JD.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 7617
B cell                 1595
Macrophage             1437
Dendritic cell         1314
Monocyte                996
Neutrophil              789
Natural killer cell     578
Smooth muscle cell      332
Endothelial cell        257
Fibroblast              187
Mast cell                49
Name: count, dtype: int64

In [87]:
data_2_ZDZJ.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 7283
B cell                 6426
Macrophage              580
Monocyte                434
Neutrophil              428
Natural killer cell     291
Fibroblast              233
Endothelial cell        144
Dendritic cell           71
Name: count, dtype: int64

In [88]:
data_AAA.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 3687
Neutrophil             1820
Macrophage              766
Fibroblast              709
Natural killer cell     486
B cell                  424
Monocyte                395
Endothelial cell        279
Pericyte                180
Mast cell                20
Dendritic cell           13
Name: count, dtype: int64

In [89]:
data_AAA_1_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell              6926
Neutrophil          2195
B cell               934
Fibroblast           749
Macrophage           352
Dendritic cell       167
Mast cell            123
Endothelial cell     105
Name: count, dtype: int64

In [90]:
data_AAA_8.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 4198
B cell                 2178
Neutrophil             1749
Fibroblast              576
Natural killer cell     301
Macrophage              253
Endothelial cell        238
Dendritic cell          195
Monocyte                148
Smooth muscle cell       61
Mast cell                24
Name: count, dtype: int64

In [91]:
data_AAA_D.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                5083
B cell                2815
Macrophage            1092
Monocyte               698
Neutrophil             639
Endothelial cell       608
Smooth muscle cell     305
Dendritic cell         228
Fibroblast              60
Name: count, dtype: int64

#### Iliac Artery Aneurysm (IAA)

In [92]:
path_AIOD_3LIB = os.path.join("../../output_data/measure_data/AIOD-3LIB/output/AIOD-3LIB_annot_add.h5ad")
path_IA_1_3LIB = os.path.join("../../output_data/measure_data/IA-1-3LIB/output/IA-1-3LIB_annot_add.h5ad")

In [93]:
data_AIOD_3LIB= sc.read_h5ad(path_AIOD_3LIB)
data_IA_1_3LIB = sc.read_h5ad(path_IA_1_3LIB)

In [94]:
data_AIOD_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                7044
Fibroblast            1638
B cell                1167
Endothelial cell       724
Monocyte               681
Macrophage             555
Smooth muscle cell     462
Neutrophil             216
Mast cell              160
Dendritic cell          32
Name: count, dtype: int64

In [95]:
data_IA_1_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
B cell                4798
T cell                3543
Endothelial cell       632
Macrophage             595
Neutrophil             420
Smooth muscle cell     273
Monocyte               171
Fibroblast              94
Mast cell               25
Name: count, dtype: int64

#### Carotid Body Tumor (CBT)

In [96]:
path_CBT = os.path.join("../../output_data/measure_data/CBT/output/CBT_annot_add.h5ad")

In [97]:
data_CBT= sc.read_h5ad(path_CBT)

In [98]:
data_CBT.obs["cell_type_level1"].value_counts()

cell_type_level1
Macrophage               3056
Endothelial cell         1576
T cell                    946
Smooth muscle cell        898
Neutrophil                769
Monocyte                  310
Fibroblast                245
Natural killer cell       245
B cell                    225
Erythrocyte/Erythroid      83
Mast cell                  76
Name: count, dtype: int64

#### Atherosclerosis

In [99]:
path_AS_CFA = os.path.join("../../output_data/measure_data/AS-CFA/output/AS-CFA_annot_add.h5ad")
path_CFA_BWD = os.path.join("../../output_data/measure_data/CFA-BWD/output/CFA-BWD_annot_add.h5ad")
path_SFA_8 = os.path.join("../../output_data/measure_data/SFA-8/output/SFA-8_annot_add.h5ad")
path_MXT_matrix = os.path.join("../../output_data/measure_data/MXT_matrix/output/MXT_matrix_annot_add.h5ad")

In [100]:
data_AS_CFA = sc.read_h5ad(path_AS_CFA)
data_CFA_BWD = sc.read_h5ad(path_CFA_BWD)
data_SFA_8 = sc.read_h5ad(path_SFA_8)
data_MXT_matrix = sc.read_h5ad(path_MXT_matrix)

In [101]:
data_AS_CFA.obs["cell_type_level1"].value_counts()

cell_type_level1
Neutrophil             3102
T cell                 2042
Natural killer cell    1108
Monocyte                710
Endothelial cell        442
Smooth muscle cell      333
Macrophage              295
Pericyte                220
B cell                  148
Fibroblast               71
Mast cell                53
Dendritic cell           28
Name: count, dtype: int64

In [102]:
data_CFA_BWD.obs["cell_type_level1"].value_counts()

cell_type_level1
Neutrophil            2331
T cell                2295
Monocyte              1504
Macrophage             575
Smooth muscle cell     268
Endothelial cell       161
B cell                 142
Dendritic cell          72
Fibroblast              46
Name: count, dtype: int64

In [103]:
data_SFA_8.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                2360
Macrophage            2109
Smooth muscle cell    1691
Fibroblast             433
Neutrophil             360
Pericyte               347
Dendritic cell         302
B cell                 110
Endothelial cell        25
Name: count, dtype: int64

In [104]:
data_MXT_matrix.obs["cell_type_level1"].value_counts()

cell_type_level1
Endothelial cell      4835
Smooth muscle cell    1415
Fibroblast             334
Macrophage             213
Pericyte                58
Mast cell               47
B cell                  33
Name: count, dtype: int64

#### Atherosclerosis, In-Stent Restenosis

In [105]:
# path_IAISR = os.path.join("/home/lixiangyu/zr/Annotate/output_data/IAISR/output/IAISR_annot.h5ad")
path_ISR_7_1 = os.path.join("../../output_data/measure_data/ISR-7-1/output/ISR-7-1_annot_add.h5ad")
path_POP_ISR_A = os.path.join("../../output_data/measure_data/POP-ISR-A/output/POP-ISR-A_annot_add.h5ad")
path_ISR_8 = os.path.join("../../output_data/measure_data/ISR-8/output/ISR-8_annot_add.h5ad")
path_LW_matrix = os.path.join("../../output_data/measure_data/LW_matrix/output/LW_matrix_annot_add.h5ad")
path_ZNB_matrix = os.path.join("../../output_data/measure_data/ZNB_matrix/output/ZNB_matrix_annot_add.h5ad")

In [106]:
# data_IAISR= sc.read_h5ad(path_IAISR)
data_ISR_7_1 = sc.read_h5ad(path_ISR_7_1)
data_POP_ISR_A = sc.read_h5ad(path_POP_ISR_A)
data_ISR_8 = sc.read_h5ad(path_ISR_8)
data_LW_matrix= sc.read_h5ad(path_LW_matrix)
data_ZNB_matrix = sc.read_h5ad(path_ZNB_matrix)

In [107]:
# data_IAISR.obs["cell_type_level1"].value_counts()

In [108]:
data_ISR_7_1.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 4309
Natural killer cell    1662
Neutrophil             1244
Macrophage              697
Monocyte                438
Dendritic cell          433
B cell                  256
Endothelial cell        255
Fibroblast              224
Mast cell                65
Smooth muscle cell       29
Name: count, dtype: int64

In [109]:
data_POP_ISR_A.obs["cell_type_level1"].value_counts()


cell_type_level1
Neutrophil             2869
T cell                 1905
Fibroblast             1633
Endothelial cell       1342
Macrophage              767
Natural killer cell     665
Monocyte                591
Pericyte                531
Dendritic cell          327
Mast cell               138
B cell                   36
Name: count, dtype: int64

In [110]:
data_ISR_8.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 2043
Macrophage              758
Monocyte                464
Neutrophil              368
Natural killer cell     313
Dendritic cell          204
Fibroblast              156
Pericyte                145
Endothelial cell        128
B cell                   38
Mast cell                24
Smooth muscle cell       22
Name: count, dtype: int64

In [111]:
data_LW_matrix.obs["cell_type_level1"].value_counts()

cell_type_level1
Neutrophil               2708
Pericyte                 1097
Smooth muscle cell        645
Macrophage                302
Fibroblast                301
Endothelial cell          253
Erythrocyte/Erythroid      49
Name: count, dtype: int64

In [112]:
data_ZNB_matrix.obs["cell_type_level1"].value_counts()

cell_type_level1
Endothelial cell      2135
Macrophage            1437
Smooth muscle cell    1068
Fibroblast             819
Pericyte               615
Neutrophil             419
Monocyte               330
T cell                 302
Mast cell              136
B cell                  33
Dendritic cell          22
Name: count, dtype: int64

#### Thoracic Aortic Aneurys (TAA)

In [113]:
path_TAA_A1_3_3LIB = os.path.join("../../output_data/measure_data/TAA-A1-3-3LIB/output/TAA-A1-3-3LIB_annot_add.h5ad")
path_TAA_Z1 = os.path.join("../../output_data/measure_data/TAA-Z1/output/TAA-Z1_annot_add.h5ad")
path_TAD_AD1_2_3LIB = os.path.join("../../output_data/measure_data/TAD-AD1-2-3LIB/output/TAD-AD1-2-3LIB_annot_add.h5ad")
path_TAD1_Z3_3LIB = os.path.join("../../output_data/measure_data/TAD1-Z3-3LIB/output/TAD1-Z3-3LIB_annot_add.h5ad")

In [114]:
data_TAA_A1_3_3LIB= sc.read_h5ad(path_TAA_A1_3_3LIB)
data_TAA_Z1 = sc.read_h5ad(path_TAA_Z1)
data_TAD_AD1_2_3LIB= sc.read_h5ad(path_TAD_AD1_2_3LIB)
data_TAD1_Z3_3LIB = sc.read_h5ad(path_TAD1_Z3_3LIB)

In [115]:
data_TAA_A1_3_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 7587
Macrophage             2138
Natural killer cell    1014
Neutrophil              567
Fibroblast              469
B cell                  277
Smooth muscle cell      268
Dendritic cell          115
Endothelial cell         73
Mast cell                20
Name: count, dtype: int64

In [116]:
data_TAA_Z1.obs["cell_type_level1"].value_counts()

cell_type_level1
Neutrophil             1244
Macrophage              776
T cell                  692
Monocyte                376
Natural killer cell     137
B cell                   93
Fibroblast               81
Mast cell                46
Dendritic cell           45
Name: count, dtype: int64

In [117]:
data_TAD_AD1_2_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 4662
Neutrophil             3813
B cell                 2771
Natural killer cell    2761
Smooth muscle cell      587
Macrophage              254
Monocyte                224
Mast cell                13
Name: count, dtype: int64

In [118]:
data_TAD1_Z3_3LIB.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                 3623
Neutrophil             2955
Macrophage              935
Natural killer cell     437
Smooth muscle cell      151
B cell                   51
Dendritic cell           46
Name: count, dtype: int64

## Healthy


In [119]:
path_cellxgene = os.path.join("../../output_data/public_data/cellxgene/cellxgene_annot_add.h5ad")
path_Bleckwehl = os.path.join("../../output_data/public_data/Bleckwehl_et_al/Bleckwehl_et_al_annot_add.h5ad")

In [120]:
data_cellxgene= sc.read_h5ad(path_cellxgene)
data_Bleckwehl = sc.read_h5ad(path_Bleckwehl)

In [121]:
data_cellxgene.obs["cell_type_level1"].value_counts()

cell_type_level1
Fibroblast               9227
Smooth muscle cell       8305
Macrophage               5236
Endothelial cell         4991
T cell                   2634
Neutrophil               1894
Pericyte                  887
Monocyte                  831
Basophil                  338
B cell                    182
Erythrocyte/Erythroid      73
Name: count, dtype: int64

In [122]:
data_Bleckwehl.obs["cell_type_level1"].value_counts()

cell_type_level1
Fibroblast               19373
Smooth muscle cell       17031
Macrophage                9489
T cell                    5350
Endothelial cell          4375
Pericyte                  1487
Natural killer cell       1192
Dendritic cell             775
Mast cell                  459
Erythrocyte/Erythroid      232
B celll                     36
Name: count, dtype: int64

In [123]:
adata_list = [data_1_JD, data_2_ZDZJ, data_AAA, data_AAA_1_3LIB, data_AAA_8, data_AAA_D,
              data_AIOD_3LIB, data_IA_1_3LIB,
              data_CBT,
              data_AS_CFA, data_CFA_BWD, data_SFA_8, data_MXT_matrix,
              data_ISR_7_1,data_POP_ISR_A, data_ISR_8, data_LW_matrix, data_ZNB_matrix,
              data_TAA_A1_3_3LIB, data_TAA_Z1, data_TAD_AD1_2_3LIB, data_TAD1_Z3_3LIB,
              data_Bleckwehl
]##data_IAISR,data_cellxgene,

In [124]:
adata_list

[AnnData object with n_obs × n_vars = 15151 × 21836
     obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
     var: 'gene_ids', 'feature_types', 'scDblFinder.selected', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'X_name', 'cell_type_level1_colors', 'decontX', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'scDblFinder.class_colors', 'scDblFinder.threshold', 'umap'
     obsm: 'X_pca', 'X_umap', 'decontX_UMAP'
     varm: 'PCs'
     layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
     obsp: 'connectivities', 'distances',
 AnnData object with n_obs × n_vars = 15

In [125]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

In [126]:
adata_final.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                   83497
Neutrophil               31005
Macrophage               29431
Fibroblast               28431
Smooth muscle cell       25839
B cell                   24550
Endothelial cell         18587
Natural killer cell      11190
Monocyte                  8470
Pericyte                  4680
Dendritic cell            4389
Mast cell                 1478
Erythrocyte/Erythroid      364
B celll                     36
Name: count, dtype: int64

In [127]:
# rename
adata_final.obs["cell_type_level1"] = adata_final.obs["cell_type_level1"].replace("B celll", "B cell")
# adata_final.obs["cell_type_level1"] = adata_final.obs["cell_type_level1"].replace("Mase cell", "Mast cell")

In [128]:
#no neurons
adata_final.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                   83497
Neutrophil               31005
Macrophage               29431
Fibroblast               28431
Smooth muscle cell       25839
B cell                   24586
Endothelial cell         18587
Natural killer cell      11190
Monocyte                  8470
Pericyte                  4680
Dendritic cell            4389
Mast cell                 1478
Erythrocyte/Erythroid      364
Name: count, dtype: int64

In [129]:
adata_final

AnnData object with n_obs × n_vars = 271947 × 35321
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    obsm: 'X_pca', 'X_umap', 'decontX_UMAP', 'Harmony', 'decontX_S1_UMAP', 'decontX_S2_UMAP', 'decontX_S3_UMAP'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [130]:
# adata_final.write_h5ad("./output_1226/small-concat.h5ad")
adata_final.write_h5ad("./output_1226/small-concat_noCxG.h5ad")

In [131]:
adata_final.var

""
A1BG
A1BG-AS1
A1CF
A2M
A2M-AS1
...
ZXDC
ZYG11A
ZYG11B
ZYX


# 2.1 Gene name mapping

In [132]:
# adata_final = sc.read_h5ad("./output_1226/small-concat.h5ad")
adata_final = sc.read_h5ad("./output_1226/small-concat_noCxG.h5ad")

In [133]:
ensembl_id_df = pd.read_csv("../../gene_names_to_ensembl_ALLFOUND_allfernandez_no6_withallslysz.csv")

In [134]:
# Create a mapping from gene names to Ensembl IDs
gene_to_ensembl = dict(zip(ensembl_id_df['gene_name'], ensembl_id_df['ensembl_id']))

In [135]:
# Map the variable names in AnnData
adata_final.var['original_gene_names'] = adata_final.var_names
adata_final.var_names = [gene_to_ensembl[gene] if gene in gene_to_ensembl else gene for gene in adata_final.var_names]

In [136]:
non_ENSG_vars = adata_final.var_names[~adata_final.var_names.str.startswith('ENSG')]
len(non_ENSG_vars)

6345

In [137]:
#save .X in a new layer
adata_final.layers["log1p_scran_samplewise"] = adata_final.X.copy() ##adata 的原始表达矩阵备份到 adata_final 的一个新层中
# take the rounded_corrected_counts layer as .X
adata_final.X = adata_final.layers["rounded_corrected_counts"].copy()###将 adata 中经过处理的表达矩阵更新为新的主要表达矩阵

In [138]:
# adata_final.write_h5ad("./output_1226/small-concat-ensembl.h5ad")
adata_final.write_h5ad("./output_1226/small-concat-ensembl-noCxG.h5ad")

In [139]:
adata_final.var

,original_gene_names
ENSG00000121410,A1BG
ENSG00000268895,A1BG-AS1
ENSG00000148584,A1CF
ENSG00000175899,A2M
ENSG00000245105,A2M-AS1
...,...
ENSG00000070476,ZXDC
ENSG00000203995,ZYG11A
ENSG00000162378,ZYG11B
ENSG00000159840,ZYX


In [140]:
# # Duplicate cells (before hvg)
# print(adata_final.layers["uncorrected_counts"].toarray().shape)
# print(np.unique(adata_final.layers["uncorrected_counts"].toarray(), axis=0).shape)
## no toarray
from scipy.sparse import csr_matrix
print(adata_final.layers["uncorrected_counts"].shape)
X = adata_final.layers["uncorrected_counts"]
X = X.tocsr()
row_signatures = set(
    zip(
        map(tuple, np.split(X.indices, X.indptr[1:-1])),
        map(tuple, np.split(X.data, X.indptr[1:-1]))
    )
)
print((len(row_signatures), X.shape[1]))


(271947, 35321)
(271947, 35321)


# 2.2 Gene names aggregation

In [77]:
# adata = sc.read_h5ad("./output_1226/small-concat-ensembl.h5ad") 
adata = sc.read_h5ad("./output_1226/small-concat-ensembl-noCxG.h5ad") 

In [54]:
#aggregation
# Convert the sparse matrix (if it is sparse) to a dense DataFrame
adata_df = pd.DataFrame(adata.X.toarray() if issparse(adata.X) else adata.X, 
                        index=adata.obs_names, 
                        columns=adata.var_names)

# Group by gene names and sum the counts
aggregated_data = adata_df.groupby(adata_df.columns, axis=1).sum()

# Prepare the new 'var' DataFrame, keeping the first occurrence of each gene
unique_var = adata.var.loc[~adata.var.index.duplicated(keep='first')]

# Create a new AnnData object with aggregated data
adata_agg = anndata.AnnData(X=aggregated_data, obs=adata.obs, var=unique_var.loc[aggregated_data.columns])

# 'adata_agg' now has unique gene names and aggregated counts

In [ ]:
adata

In [ ]:
adata_agg

In [ ]:
# MANUALLY CHECK IF SUMMATION WORKED AS INTENDED(做验证)
# Extract counts for "ENSG00000033050" from the original adata
original_counts = adata[:, "ENSG00000033050"].X
if issparse(original_counts):
    original_counts = original_counts.toarray()  # Convert to dense array if sparse

# Sum these counts cellwise
summed_counts = np.sum(original_counts, axis=1)

# Extract counts for "ENSG00000033050" from adata_agg
agg_counts = adata_agg[:, "ENSG00000033050"].X
if issparse(agg_counts):
    agg_counts = agg_counts.toarray()  # Convert to dense array if sparse

# Compare the two sets of counts
comparison = np.allclose(summed_counts.flatten(), agg_counts.flatten())

# Print the result of the comparison
print(f"The counts are correctly summed: {comparison}")


In [58]:
adata_agg.write_h5ad("./output_1226/small-concat-aggr-X.h5ad")

In [ ]:
adata_agg = sc.read_h5ad("./output_1226/small-concat-aggr-X.h5ad") 

In [ ]:
# # Function to aggregate a layer
# def aggregate_layer(layer):
#     layer_df = pd.DataFrame(layer.toarray() if issparse(layer) else layer, 
#                             index=adata.obs_names, 
#                             columns=adata.var_names)
#     return layer_df.groupby(layer_df.columns, axis=1).sum()

# # Aggregate each layer and add to adata_agg
# for layer_name in adata.layers.keys():
#     print(f"Aggregating layer: {layer_name}")
#     aggregated_layer = aggregate_layer(adata.layers[layer_name])
    
#     # The first time, we need to initialize layers in adata_agg
#     if not hasattr(adata_agg, 'layers'):
#         adata_agg.layers = {}

#     # Add the aggregated layer to adata_agg
#     adata_agg.layers[layer_name] = aggregated_layer

# # Now 'adata_agg' contains all the aggregated layers
#####.py

In [3]:
# adata = sc.read_h5ad("./output_1226/small-concat-ensembl.h5ad") 
# adata_agg = sc.read_h5ad("./output_1226/small-concat-ensembl-aggr.h5ad") 
adata = sc.read_h5ad("./output_1226/small-concat-ensembl-noCxG.h5ad") 
adata_agg = sc.read_h5ad("./output_1226/small-concat-ensembl-aggr-noCxG.h5ad") 

In [4]:
adata_agg

AnnData object with n_obs × n_vars = 271947 × 34680
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    var: 'original_gene_names'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [5]:
adata

AnnData object with n_obs × n_vars = 271947 × 35321
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    var: 'original_gene_names'
    obsm: 'Harmony', 'X_pca', 'X_umap', 'decontX_S1_UMAP', 'decontX_S2_UMAP', 'decontX_S3_UMAP', 'decontX_UMAP'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [6]:
def sum_gene_counts(layer, gene_name, var_names):
    # Find the index(es) of the gene
    gene_indices = np.where(var_names == gene_name)[0]

    # Sum counts across all occurrences of the gene
    gene_counts = np.sum(layer[:, gene_indices].toarray(), axis=1) if issparse(layer) else np.sum(layer[:, gene_indices], axis=1)
    return gene_counts

# Check for each layer
for layer_name in adata.layers.keys():
    print(f"Checking layer: {layer_name}")

    # Get var names for the current layer
    var_names = adata.var_names

    # Sum counts for "ENSG00000033050" in the original data
    original_counts = sum_gene_counts(adata.layers[layer_name], "ENSG00000033050", var_names)

    # Extract counts for "ENSG00000033050" from adata_agg layer
    gene_index_agg = np.where(adata_agg.var_names == "ENSG00000033050")[0]
    agg_counts = adata_agg.layers[layer_name][:, gene_index_agg]
    if issparse(agg_counts):
        agg_counts = agg_counts.toarray()

    # Compare the two sets of counts
    comparison = np.allclose(original_counts.flatten(), agg_counts.flatten())

    # Print the result of the comparison
    print(f"The counts are correctly summed in layer {layer_name}: {comparison}")

Checking layer: log1p_scran_samplewise
The counts are correctly summed in layer log1p_scran_samplewise: True
Checking layer: raw_decontXcounts
The counts are correctly summed in layer raw_decontXcounts: True
Checking layer: rounded_corrected_counts
The counts are correctly summed in layer rounded_corrected_counts: True
Checking layer: uncorrected_counts
The counts are correctly summed in layer uncorrected_counts: True


In [7]:
adata_agg

AnnData object with n_obs × n_vars = 271947 × 34680
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    var: 'original_gene_names'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [8]:
import scipy.sparse as sparse

In [9]:
adata_agg.X = sparse.csr_matrix(adata_agg.X)
for layer_name in adata_agg.layers.keys():
    print("converting to sparse matrix in", layer_name)
    adata_agg.layers[layer_name] = sparse.csr_matrix(adata_agg.layers[layer_name])


converting to sparse matrix in log1p_scran_samplewise
converting to sparse matrix in raw_decontXcounts
converting to sparse matrix in rounded_corrected_counts
converting to sparse matrix in uncorrected_counts


In [10]:
# adata_agg.write_h5ad("./output_1226/small-concat-aggr-sparse.h5ad")
adata_agg.write_h5ad("./output_1226/small-concat-aggr-sparse-noCxG.h5ad")

# 3. Normalization

In [11]:
# adata_final = sc.read_h5ad("./output_1226/small-concat-aggr-sparse.h5ad")
adata_final = sc.read_h5ad("./output_1226/small-concat-aggr-sparse-noCxG.h5ad")

In [12]:
#Perform a clustering for scran normalization in clusters
adata_pp = adata_final.copy()
sc.pp.normalize_total(adata_pp, target_sum=1e6)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, svd_solver="arpack")
sc.pp.neighbors(adata_pp, n_pcs=30)
sc.tl.leiden(adata_pp, key_added='groups', resolution=0.22)

In [13]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

In [14]:
#Preprocess variables for scran normalization
input_groups = adata_pp.obs['groups']
# data_mat = adata_final.X.T.toarray()
data_mat = adata_final.X.T 

In [15]:
%%R -i data_mat -i input_groups -o size_factors
library(scran)
library(Matrix)  # 必须加载此包以支持稀疏矩阵
print("转换稀疏格式")
data_mat <- as(data_mat, "dgCMatrix")  # 从Python的CSC转为R的CSC格式
size_factors <- calculateSumFactors(data_mat, clusters=input_groups, min.mean=0.1)
print("计算结束")

[1] "转换稀疏格式"
[1] "计算结束"


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [16]:
del adata_pp
adata_final.obs['size_factors'] = size_factors

In [17]:
#Normalize adata 
adata_final.X /= adata_final.obs['size_factors'].values[:,None]
sc.pp.log1p(adata_final)

In [18]:
# adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-scranlog1p.h5ad")
adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-scranlog1p-noCxG.h5ad")

# 4. Integrate

In [3]:
# adata_final = sc.read_h5ad("./output_1226/small-concat-aggr-sparse-scranlog1p.h5ad")
adata_final = sc.read_h5ad("./output_1226/small-concat-aggr-sparse-scranlog1p-noCxG.h5ad")

In [4]:
def prepare_adata(adata_final):
    
    #delete all layers that are not needed to reduce memory
    del adata_final.layers['log1p_scran_samplewise'] #only for small integrations
    del adata_final.layers['raw_decontXcounts']
    del adata_final.layers['uncorrected_counts']

    #rename layer to counts
    adata_final.layers['counts'] = adata_final.layers['rounded_corrected_counts']
    del adata_final.layers['rounded_corrected_counts']

    # add "_{dataset}" to the sample name to make them unique
    adata_final.obs["sample"] = adata_final.obs["sample"].astype(str) + "_" + adata_final.obs["dataset"].astype(str)
    adata_final.obs['sample'] = adata_final.obs['sample'].astype('category')   

In [5]:
def preprocess_adata(adata_final, mode, batch_key):

    # remove unknown and doublets
    print("Removing unknowns and doublets...")
    adata_final = adata_final[~adata_final.obs['cell_type_level1'].isin(['unknown', 'doublets'])].copy()

    
    if mode=="auto":
        print("Using auto mode to do hvg and pca...")
        scib.preprocessing.reduce_data(adata_final, batch_key=batch_key)
        
    elif mode=="manual":
        print("Using manual mode to do hvg and pca...")
        sc.pp.highly_variable_genes(adata_final,  n_top_genes = 2000, batch_key = batch_key)
        sc.pp.pca(adata_final, use_highly_variable=True)

    else:
        raise ValueError("Mode must be 'auto' or 'manual'")

    
    # subset only hvgs
    print("Subset for HVGs...")
    hvg = adata_final.var[adata_final.var['highly_variable']].index.tolist()
    adata_final = adata_final[:, hvg].copy()

    
    #check how many cells have zero counts for all genes
    print("Check how many cells have zero counts for all genes...")
    cellwise_sum = adata_final.X.sum(axis=1)
    num_cells_zero_counts = (cellwise_sum == 0).sum()
    
    if num_cells_zero_counts>0:
        print(num_cells_zero_counts, " cells were found with 0 counts across all genes! Removing these cells now...")
        adata_final = adata_final[cellwise_sum > 0, :]

    
    # check for dublicate gene expressions across cells after HVG selection
    print("Check for dublicate gene expressions")
    before = adata_final.layers["counts"].toarray().shape[0]
    after = np.unique(adata_final.layers["counts"].toarray(), axis=0).shape[0]
    diff = before - after

    if diff > 0:
        print(diff, " Non unique cell expression profiles found! Removing them...")
        
        counts_array = adata_final.layers["counts"].toarray()
        # Find the indices of unique rows
        _, unique_indices = np.unique(counts_array, axis=0, return_index=True)
        # Sort the indices to maintain the original order
        unique_indices_sorted = np.sort(unique_indices)
        # Filter the AnnData object to keep only unique rows
        adata_final = adata_final[unique_indices_sorted]
    else:
        "No non unique cells found."

    print("Preprocessing finished!")  

    return adata_final

In [6]:
def integrate_methods(adata, batch_key, cell_type, scgen_impl):
    
    # adata has to be filtered, hvg filtered and pca applied in obsm[X_pca]
    # log normalized data in .X and counts in .layers["counts"] 

    
    adata_final = adata

    # scGen
    print("Integrating with scGen...")

    if scgen_impl == "scib":
        print("scGen scib implementation")
        adata_after = scib.ig.scgen(adata_final, batch=batch_key, cell_type=cell_type)
        adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
        
    elif scgen_impl == "scarches":
        print("scGen scarches implementation")

        adata_final = remove_sparsity(adata_final) # remove sparsity
        adata_final.X = adata_final.X.astype("float32")
        
        epoch = 100
        early_stopping_kwargs = {
            "early_stopping_metric": "val_loss",
            "patience": 25,
            "threshold": 0,
            "reduce_lr": True,
            "lr_patience": 13,
            "lr_factor": 0.1,
        }

        network = sca.models.scgen(adata = adata_final,
                           hidden_layer_sizes=[256,128])

        network.train(n_epochs=epoch, early_stopping_kwargs = early_stopping_kwargs, batch_size = 32)

        adata_after = network.batch_removal(adata_final, batch_key=batch_key, cell_label_key=cell_type,return_latent=True)
        adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    
    
    # scVI
    print("Integrating with scVI...")
    
    scvi.model.SCVI.setup_anndata(adata_final, layer="counts", batch_key=batch_key)
    vae = scvi.model.SCVI(adata_final, gene_likelihood="nb", n_layers=2, n_latent=30)
    vae.train()
    adata_final.obsm["scVI"] = vae.get_latent_representation()

    
    # scANVI
    print("Integrating with scANVI...")
    
    lvae = scvi.model.SCANVI.from_scvi_model(
        vae,
        #adata=adata_final,
        labels_key=cell_type,
        unlabeled_category="none"
    )
    lvae.train(max_epochs=40)
    adata_final.obsm["scANVI"] = lvae.get_latent_representation()

    
    # scPoli
    print("Integrating with scPoli...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model = scPoli(
    adata=adata_final,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='nb',
    )
    
    scpoli_model.train(
        n_epochs=50,
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,
    )

    scpoli_latent = scpoli_model.get_latent(adata_final)
    adata_final.obsm["scPoli"] = scpoli_latent

    
    # Harmony
    print("Integrating with Harmony...")
    adata_final.obsm["Harmony"] = harmonize(adata_final.obsm["X_pca"], adata_final.obs, batch_key=batch_key)

    
    # LIGER
    print("Integrating with LIGER...")

    batch_cats = adata_final.obs["sample"].cat.categories
    bdata = adata_final.copy()
    # Pyliger normalizes by library size with a size factor of 1
    # So here we give it the count data
    bdata.X = bdata.layers["counts"]
    # List of adata per dataset
    adata_list = [bdata[bdata.obs[batch_key] == b].copy() for b in batch_cats]
    for i, ad in enumerate(adata_list):
        ad.uns["sample_name"] = batch_cats[i]
        # Hack to make sure each method uses the same genes
        ad.uns["var_gene_idx"] = np.arange(bdata.n_vars)
    
    
    liger_data = pyliger.create_liger(adata_list, remove_missing=False, make_sparse=False)
    # Hack to make sure each method uses the same genes
    liger_data.var_genes = bdata.var_names
    pyliger.normalize(liger_data)
    pyliger.scale_not_center(liger_data)
    pyliger.optimize_ALS(liger_data, k=30)
    pyliger.quantile_norm(liger_data)
    
    
    adata_final.obsm["LIGER"] = np.zeros((adata_final.shape[0], liger_data.adata_list[0].obsm["H_norm"].shape[1]))
    for i, b in enumerate(batch_cats):
        adata_final.obsm["LIGER"][adata_final.obs[batch_key] == b] = liger_data.adata_list[i].obsm["H_norm"]

    print("Finished integrating the data")
    
    return adata_final

In [7]:
def integrate_sca(adata, batch_key, cell_type,):
    
    # adata has to be filtered, hvg filtered and pca applied in obsm[X_pca]
    # log normalized data in .X and counts in .layers["counts"] 

    
    adata_final = adata

    adata_counts = adata_final.copy()
    adata_counts.X = adata_counts.layers["counts"].copy()

    # print("scGen scib implementation")
    # adata_after = scib.ig.scgen(adata_final, batch=batch_key, cell_type=cell_type)
    # adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    
    '''
    # scGen too slow
    print("Integrating with scGen...")

    adata_final = remove_sparsity(adata_final) # remove sparsity
    adata_final.X = adata_final.X.astype("float32")
    
    epoch = 100
    early_stopping_kwargs = {
        "early_stopping_metric": "val_loss",
        "patience": 25,
        "threshold": 0,
        "reduce_lr": True,
        "lr_patience": 13,
        "lr_factor": 0.1,
    }

    network = sca.models.scgen(adata = adata_final,
                       hidden_layer_sizes=[256,128])

    network.train(n_epochs=epoch, early_stopping_kwargs = early_stopping_kwargs, batch_size = 32)

    adata_after = network.batch_removal(adata_final, batch_key=batch_key, cell_label_key=cell_type,return_latent=True)
    adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    '''

    
    # scVI
    print("Integrating with scVI...")

    sca.models.SCVI.setup_anndata(adata_counts, batch_key=batch_key)

    vae = sca.models.SCVI(
        adata_counts,
        n_layers=2,##默认为1
        encode_covariates=True,
        deeply_inject_covariates=False,
        use_layer_norm="both",
        use_batch_norm="none",
    )

    vae.train()
    adata_final.obsm["scVI"] = vae.get_latent_representation()

    
    # scANVI
    print("Integrating with scANVI...")

    scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "none", labels_key=cell_type)
    scanvae.train(max_epochs=40)
    adata_final.obsm["scANVI"] = scanvae.get_latent_representation()##采用训练好的scvi模型

    
    
    # scPoli
    print("Integrating with scPoli...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model = scPoli(
    adata=adata_final,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='mse',
    )
    
    scpoli_model.train(
        n_epochs=50,##默认为100
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,##默认为1
    )

    scpoli_latent = scpoli_model.get_latent(adata_final)
    adata_final.obsm["scPoli"] = scpoli_latent


    '''
    ####################################################################################
    # scPoli counts
    print("Integrating with scPoli using raw counts...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model2 = scPoli(
    adata=adata_counts,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='nb',
    )
    
    scpoli_model2.train(
        n_epochs=50,
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,
    )

    scpoli_latent2 = scpoli_model2.get_latent(adata_counts)
    adata_final.obsm["scPoli_counts"] = scpoli_latent2
    ####################################################################################
    '''
    
    # Harmony
    print("Integrating with Harmony...")
    adata_final.obsm["Harmony"] = harmonize(adata_final.obsm["X_pca"], adata_final.obs, batch_key=batch_key)

    
    # LIGER
    print("Integrating with LIGER...")

    batch_cats = adata_final.obs["sample"].cat.categories
    bdata = adata_final.copy()
    # Pyliger normalizes by library size with a size factor of 1
    # So here we give it the count data
    bdata.X = bdata.layers["counts"]
    # List of adata per dataset
    adata_list = [bdata[bdata.obs[batch_key] == b].copy() for b in batch_cats]
    for i, ad in enumerate(adata_list):
        ad.uns["sample_name"] = batch_cats[i]
        # Hack to make sure each method uses the same genes
        ad.uns["var_gene_idx"] = np.arange(bdata.n_vars)
    
    
    liger_data = pyliger.create_liger(adata_list, remove_missing=False, make_sparse=False)
    # Hack to make sure each method uses the same genes
    liger_data.var_genes = bdata.var_names
    pyliger.normalize(liger_data)
    pyliger.scale_not_center(liger_data)
    pyliger.optimize_ALS(liger_data, k=30)
    pyliger.quantile_norm(liger_data)
    
    
    adata_final.obsm["LIGER"] = np.zeros((adata_final.shape[0], liger_data.adata_list[0].obsm["H_norm"].shape[1]))
    for i, b in enumerate(batch_cats):
        adata_final.obsm["LIGER"][adata_final.obs[batch_key] == b] = liger_data.adata_list[i].obsm["H_norm"]


    
    print("Finished integrating the data")
    
    return adata_final

In [8]:
import time
start_time = time.time()

In [9]:
prepare_adata(adata_final)

In [ ]:
adata_final

In [ ]:
adata_final = preprocess_adata(adata_final, mode = "auto", batch_key="sample")

In [ ]:
adata_final = integrate_sca(adata_final, batch_key = "sample", cell_type = "cell_type_level1")##no-scgen(版本问题)

In [ ]:
end_time = time.time()
print(f"Elapsed time: {end_time - start_time} seconds")

In [40]:
# check for dublicates in embeddings

In [ ]:
for embed in ['Harmony', 'LIGER', 'X_pca', 'scANVI', 'scPoli', 'scVI']:
    diff = adata_final.obsm[embed].shape[0] - np.unique(adata_final.obsm[embed], axis=0).shape[0]
    print(diff," for ", embed )

In [13]:
#rename obsm X_pca to PCA in adata_final
adata_final.obsm["PCA"] = adata_final.obsm["X_pca"]
del adata_final.obsm["X_pca"]

In [14]:
# adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-method-mse.h5ad") # used in the 3-1_evaluation-HPC.py script
adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-method-mse-noCxG.h5ad") # used in the 3-1_evaluation-HPC.py script

# UMAP-dim-5

In [33]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse.h5ad')

## harmony

In [34]:
adata.obsm['X_pca'] = adata.obsm['Harmony']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="harmony_cell_type1",save='_cell_type_dim5-harmony.png')
sc.pl.umap(adata, color='sample',title="harmony_sample",save='_sample_dim5-harmony.png')

## scANVI

In [35]:
adata.obsm['X_pca'] = adata.obsm['scANVI']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scANVI_cell_type1",save='_cell_type_dim5-scANVI.png')
sc.pl.umap(adata, color='sample',title="scANVI_sample",save='_sample_dim5-scANVI.png')

## scVI

In [36]:
adata.obsm['X_pca'] = adata.obsm['scVI']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scVI_cell_type1",save='_cell_type_dim5-scVI.png')
sc.pl.umap(adata, color='sample',title="scVI_sample",save='_sample_dim5-scVI.png')

## LIGER

In [37]:
adata.obsm['X_pca'] = adata.obsm['LIGER']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="LIGER_cell_type1",save='_cell_type_dim5-LIGER.png')
sc.pl.umap(adata, color='sample',title="LIGER_sample",save='_sample_dim5-LIGER.png')

## scPoli

In [38]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim5-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim5-scPoli.png')

## PCA

In [39]:
adata.obsm['X_pca'] = adata.obsm['PCA']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="PCA_cell_type1",save='_cell_type_dim5-PCA.png')
sc.pl.umap(adata, color='sample',title="PCA_sample",save='_sample_dim5-PCA.png')

# UMAP-dim=10,eta=2

In [2]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta2.h5ad')

## scPoli

In [3]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta2-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta2-scPoli.png')

# UMAP-dim=10,eta=3

In [4]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta3.h5ad')

## scPoli

In [5]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta3-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta3-scPoli.png')

# UMAP-eta=8

In [40]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-eta8.h5ad')

## scPoli

In [45]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_eta8-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_eta8-scPoli.png')

# UMAP-dim=10,eta=4

In [30]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta4.h5ad')

## scPoli

In [31]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta4-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta4-scPoli.png')

# UMAP-dim10

In [27]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10.h5ad')

## scPoli

In [28]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10-scPoli.png')

# UMAP-dim=10,eta=6

In [54]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta6.h5ad')

## scPoli

In [56]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta6-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta6-scPoli.png')